# Data Preprocessing Pipeline
 

## Imports

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_selection import VarianceThreshold
from sklearn.preprocessing import StandardScaler
from skmultilearn.model_selection import iterative_train_test_split


## Step 1 — Load Data

In [3]:
df = pd.read_csv(r"C:\Users\hp\Desktop\stage etis\baseline\goodscents_jadbio_ready.csv", sep=';')

LABEL_COLS = [
    'floral', 'fruity', 'sweet', 'woody', 'green', 'spicy',
    'animal_musk', 'earthy', 'citrus', 'chemical', 'gourmand', 'powdery_amber'
]

fp_cols      = [c for c in df.columns if c.startswith('MACCS_') or c.startswith('morgan_')]
mordred_cols = [c for c in df.columns if c not in LABEL_COLS + ['SMILES'] + fp_cols]

print(f'Total molecules      : {len(df)}')
print(f'MACCS + Morgan cols  : {len(fp_cols)}')
print(f'Mordred cols         : {len(mordred_cols)}')
print(f'Label cols           : {len(LABEL_COLS)}')

Total molecules      : 4981
MACCS + Morgan cols  : 678
Mordred cols         : 327
Label cols           : 12


## Step 2 — Inspect NaN Values

In [4]:
X_all = df[fp_cols + mordred_cols].values.astype(float)

nan_rows = np.isnan(X_all).any(axis=1).sum()
nan_cols = np.isnan(X_all).any(axis=0).sum()

nan_cols_fp      = np.isnan(df[fp_cols].values.astype(float)).any(axis=0).sum()
nan_cols_mordred = np.isnan(df[mordred_cols].values.astype(float)).any(axis=0).sum()

nan_row_indices = list(np.where(np.isnan(X_all).any(axis=1))[0])

print(f'Rows with NaN        : {nan_rows}')
print(f'Columns with NaN     : {nan_cols}')
print(f'  - in fingerprints  : {nan_cols_fp}')
print(f'  - in Mordred       : {nan_cols_mordred}')
print(f'NaN row indices      : {nan_row_indices}')
print()
print('SMILES of problematic molecules:')
for idx in nan_row_indices:
    print(f'  row {idx}: {df.iloc[idx]["SMILES"]}')

Rows with NaN        : 5
Columns with NaN     : 327
  - in fingerprints  : 0
  - in Mordred       : 327
NaN row indices      : [np.int64(3917), np.int64(3943), np.int64(3977), np.int64(3978), np.int64(4646)]

SMILES of problematic molecules:
  row 3917: CC(C)=CCCC(C)CC(OCCCCCCCCCCC(C)C)OCCCCCCCCCCC(C)C
  row 3943: CC(C)CCCCCCCCCCOC(CC(C)CCCC(C)(C)O)OCCCCCCCCCCC(C)C
  row 3977: CC1=C2CC(C=O)C(C1)C(C(C)C)C2
  row 3978: CC(=O)C1CC2=C(C)CC1C(C(C)C)C2
  row 4646: CC(C)CCCCCCCCCCCCCCC(=O)OCC(O)COCC(COC(=O)CCCCCCCCCCCCCCC(C)C)OC(=O)CCCCCCCCCCCCCCC(C)C


## Step 3 — Drop NaN Rows

Only 5 molecules (0.1% of the dataset) have NaN values — always the same 5 rows across all 327 Mordred features.
This is a Mordred computation failure on specific molecules (very large or unusual structures), not random missing data.
Given the negligible data loss, we drop these rows entirely rather than imputing.

In [5]:
df_clean = df.dropna(subset=fp_cols + mordred_cols).reset_index(drop=True)

print(f'Molecules before : {len(df)}')
print(f'Molecules after  : {len(df_clean)}')
print(f'Dropped          : {len(df) - len(df_clean)}')
print(f'NaN remaining    : {df_clean[fp_cols + mordred_cols].isna().sum().sum()}')

Molecules before : 4981
Molecules after  : 4976
Dropped          : 5
NaN remaining    : 0


## Step 4 — Split into Feature Groups

We separate features into two groups because they require different preprocessing:
- **Fingerprints** (MACCS + Morgan): binary (0/1), no scaling needed
- **Mordred**: continuous physicochemical values on different scales, requires StandardScaler

In [6]:
X_fp      = df_clean[fp_cols].values.astype(float)
X_mordred = df_clean[mordred_cols].values.astype(float)
y         = df_clean[LABEL_COLS].values

print(f'X_fingerprints shape : {X_fp.shape}')
print(f'X_mordred shape      : {X_mordred.shape}')
print(f'y shape              : {y.shape}')

X_fingerprints shape : (4976, 678)
X_mordred shape      : (4976, 327)
y shape              : (4976, 12)


## Step 5 — Remove Zero-Variance Features

Features with zero variance are identical across all molecules — they carry no information and should be removed.
This is applied independently to each feature group.

Note: this step is safe to do before the train/test split because zero-variance detection uses no statistical learning — it simply checks if a column is constant.

In [7]:
# Fingerprints
vt_fp = VarianceThreshold(threshold=0)
X_fp = vt_fp.fit_transform(X_fp)
fp_cols_kept = np.array(fp_cols)[vt_fp.get_support()]

print(f'Fingerprints: {len(fp_cols)} -> {X_fp.shape[1]} features '
      f'({len(fp_cols) - X_fp.shape[1]} zero-variance dropped)')

# Mordred
vt_mordred = VarianceThreshold(threshold=0)
X_mordred = vt_mordred.fit_transform(X_mordred)
mordred_cols_kept = np.array(mordred_cols)[vt_mordred.get_support()]

print(f'Mordred     : {len(mordred_cols)} -> {X_mordred.shape[1]} features '
      f'({len(mordred_cols) - X_mordred.shape[1]} zero-variance dropped)')
print(f'\nTotal features after zero-variance removal: {X_fp.shape[1] + X_mordred.shape[1]}')

Fingerprints: 678 -> 672 features (6 zero-variance dropped)
Mordred     : 327 -> 327 features (0 zero-variance dropped)

Total features after zero-variance removal: 999


## Step 6 — Train/Test Split

We use `iterative_train_test_split` from scikit-multilearn instead of sklearn's `train_test_split`.

**Why?** Standard stratified splitting works for single-label problems. For multi-label data with imbalanced labels, `iterative_train_test_split` preserves the positive/negative ratio of **each label independently** in both train and test sets — critical given our class imbalance.

In [8]:
# iterative_train_test_split requires a single X matrix
# We concatenate temporarily just for the split, then separate again
X_combined = np.hstack([X_fp, X_mordred])
n_fp = X_fp.shape[1]

X_train_comb, y_train, X_test_comb, y_test = iterative_train_test_split(
    X_combined, y, test_size=0.2
)

# Separate back into fingerprints and Mordred
X_fp_train    = X_train_comb[:, :n_fp]
X_mordred_train = X_train_comb[:, n_fp:]
X_fp_test     = X_test_comb[:, :n_fp]
X_mordred_test  = X_test_comb[:, n_fp:]

print(f'Train : {X_fp_train.shape[0]} molecules')
print(f'Test  : {X_fp_test.shape[0]} molecules')
print()
print('Label distribution preserved (% positive):')
print(f'{"Label":<20} {"Full":>8} {"Train":>8} {"Test":>8}')
print('-' * 48)
for i, l in enumerate(LABEL_COLS):
    full  = y[:, i].mean() * 100
    train = y_train[:, i].mean() * 100
    test  = y_test[:, i].mean() * 100
    print(f'{l:<20} {full:>7.1f}% {train:>7.1f}% {test:>7.1f}%')

Train : 3882 molecules
Test  : 1094 molecules

Label distribution preserved (% positive):
Label                    Full    Train     Test
------------------------------------------------
floral                  26.2%    26.8%    23.8%
fruity                  45.8%    47.0%    41.7%
sweet                   38.7%    39.7%    35.2%
woody                   21.2%    21.7%    19.3%
green                   45.2%    46.3%    41.1%
spicy                   21.6%    22.1%    19.7%
animal_musk             15.5%    15.9%    14.1%
earthy                  20.2%    20.7%    18.4%
citrus                  10.3%    10.6%     9.3%
chemical                43.1%    44.2%    39.2%
gourmand                24.2%    24.9%    21.8%
powdery_amber           20.6%    21.1%    18.7%


## Step 7 — Scale Mordred Features

`StandardScaler` transforms each feature to zero mean and unit variance:
$$x_{scaled} = \frac{x - \mu_{train}}{\sigma_{train}}$$

Critically, the scaler is **fitted on train only** and applied to both train and test.
Fitting on the full dataset before splitting would leak test set statistics into preprocessing — an overoptimistic evaluation.

In [9]:
scaler = StandardScaler()

# Fit on train, transform both
X_mordred_train = scaler.fit_transform(X_mordred_train)
X_mordred_test  = scaler.transform(X_mordred_test)      # same scaler, no refit

print('StandardScaler fitted on train only.')
print(f'Mordred train mean (should be ~0): {X_mordred_train.mean():.4f}')
print(f'Mordred train std  (should be ~1): {X_mordred_train.std():.4f}')
print(f'Mordred test mean  (not forced 0): {X_mordred_test.mean():.4f}')

StandardScaler fitted on train only.
Mordred train mean (should be ~0): -0.0000
Mordred train std  (should be ~1): 1.0000
Mordred test mean  (not forced 0): -0.0270


## Step 8 — Correlation Filter on Mordred (Train Only)

Highly correlated Mordred features (r > 0.95) are redundant — they encode the same information.
For each correlated pair, we drop one feature.

The correlation structure is computed on **train only** and the same column mask is applied to test.

In [10]:
# Compute correlation matrix on train
corr_matrix = np.corrcoef(X_mordred_train.T)
corr_matrix = np.abs(corr_matrix)

# Find columns to drop
upper_triangle = np.triu(corr_matrix, k=1)
cols_to_drop = set()
rows, cols = np.where(upper_triangle > 0.95)
for r, c in zip(rows, cols):
    if c not in cols_to_drop:
        cols_to_drop.add(c)

cols_to_keep = [i for i in range(X_mordred_train.shape[1]) if i not in cols_to_drop]

# Apply same mask to both train and test
X_mordred_train = X_mordred_train[:, cols_to_keep]
X_mordred_test  = X_mordred_test[:, cols_to_keep]

print(f'Mordred features before correlation filter : {len(cols_to_keep) + len(cols_to_drop)}')
print(f'Features dropped (r > 0.95)               : {len(cols_to_drop)}')
print(f'Mordred features remaining                : {X_mordred_train.shape[1]}')

Mordred features before correlation filter : 327
Features dropped (r > 0.95)               : 7
Mordred features remaining                : 320


## Step 9 — Concatenate Final Feature Matrices

Fingerprints (unscaled, binary) and Mordred (scaled, filtered) are concatenated into the final feature matrices.

In [11]:
X_train = np.hstack([X_fp_train, X_mordred_train])
X_test  = np.hstack([X_fp_test,  X_mordred_test])

print('Final feature matrices:')
print(f'  X_train : {X_train.shape}')
print(f'  X_test  : {X_test.shape}')
print(f'  y_train : {y_train.shape}')
print(f'  y_test  : {y_test.shape}')
print()
print('Preprocessing complete. Ready for model training.')

Final feature matrices:
  X_train : (3882, 992)
  X_test  : (1094, 992)
  y_train : (3882, 12)
  y_test  : (1094, 12)

Preprocessing complete. Ready for model training.
